# ⚙️ Project 4.3 — Round-Robin Task Scheduler
**DSA for Mechatronics · Week 04 — Linked Lists**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A microcontroller runs multiple sensor-polling tasks in a **round-robin** loop —
each task gets a fixed time quantum before the scheduler moves to the next.
If a task misses too many quanta, it is dropped.
The scheduler is modelled as a **circular linked list** — the tail points back to the head, so cycling is natural and there is no "end".


## Step 1 — Build the circular task list

In [ ]:
class TNode:
    """A task in the round-robin scheduler."""
    def __init__(self, task_id, name, priority, remaining):
        self.task_id   = task_id
        self.name      = name
        self.priority  = priority    # 1 (low) – 5 (high)
        self.remaining = remaining   # work units left
        self.next      = None
    def __repr__(self):
        return f"Task({self.task_id},{self.name},p={self.priority},r={self.remaining})"

# ── Personalised parameters ─────────────────────────────────────────
TASK_NAMES = ["ADC_POLL","CAN_RX","PWM_GEN","ENCODER","TEMP_MON",
              "IMU_READ","UART_TX","WATCHDOG","PID_CTRL","LOG_WRITE"]
N_TASKS    = random.randint(5, 9)
QUANTUM    = random.randint(2, 5)    # work units per turn
DROP_MISS  = random.randint(3, 6)    # drop task after this many consecutive misses
selected   = random.sample(TASK_NAMES, N_TASKS)

head = None
tail = None
for i, name in enumerate(selected):
    node = TNode(
        task_id   = i + 1,
        name      = name,
        priority  = random.randint(1, 5),
        remaining = random.randint(QUANTUM * 2, QUANTUM * 8),
    )
    if head is None:
        head = tail = node
    else:
        tail.next = node
        tail = node
tail.next = head   # close the circle

def circular_to_list(head_node, max_nodes=None):
    """Snapshot of the circular list (up to max_nodes to avoid infinite loop)."""
    if not head_node: return []
    out = [head_node]
    cur = head_node.next
    limit = max_nodes or 10000
    while cur is not head_node and len(out) < limit:
        out.append(cur)
        cur = cur.next
    return out

tasks = circular_to_list(head)
print(f"Scheduler parameters:")
print(f"  Tasks         : {N_TASKS}")
print(f"  Quantum       : {QUANTUM} work units / turn")
print(f"  Drop after    : {DROP_MISS} consecutive misses")
print()
print(f"  {'ID':>3}  {'Name':<12}  {'Priority':>8}  {'Work left':>10}")
print(f"  {'─'*3}  {'─'*12}  {'─'*8}  {'─'*10}")
for t in tasks:
    print(f"  {t.task_id:3d}  {t.name:<12}  {t.priority:8d}  {t.remaining:10d}")


## Step 2 — Simulate the scheduler (circular traversal)

In [ ]:
def simulate_scheduler(head_node, quantum, drop_miss):
    """
    Simulate round-robin scheduling on a circular linked list.
    Each pass: every task gets `quantum` work units.
    A task is removed when remaining <= 0.
    Returns schedule log and completion order.
    """
    miss_count    = {t.task_id: 0 for t in circular_to_list(head_node)}
    completed     = []
    dropped       = []
    schedule_log  = []
    tick          = 0
    current       = head_node
    prev          = None

    # find the tail for deletion
    tmp = head_node
    while tmp.next is not head_node:
        tmp = tmp.next
    tail_ptr = tmp

    n_active = len(circular_to_list(head_node))

    while n_active > 0 and tick < 200:
        tick  += 1
        node   = current
        nxt    = current.next

        work_done     = min(quantum, node.remaining)
        node.remaining -= work_done
        schedule_log.append((tick, node.task_id, node.name, work_done, node.remaining))

        if node.remaining <= 0:
            completed.append((node.task_id, node.name, tick))
            # remove from circular list
            if n_active == 1:
                head_node = None
                break
            if prev:
                prev.next = nxt
            else:
                # removing current head — find new head
                head_node = nxt
                tail_ptr.next = head_node
            n_active -= 1
            current = nxt
            continue

        prev    = node
        current = nxt

    return schedule_log, completed

log, completion_order = simulate_scheduler(head, QUANTUM, DROP_MISS)

total_ticks = log[-1][0] if log else 0
print(f"Simulation complete in {total_ticks} ticks")
print()
print("Completion order:")
for tid, name, tick in completion_order:
    print(f"  Task {tid} ({name:<12}) completed at tick {tick}")
print()
print("First 10 schedule ticks:")
print(f"  {'Tick':>5}  {'Task':>5}  {'Name':<12}  {'Work':>5}  {'Remaining':>10}")
print(f"  {'─'*5}  {'─'*5}  {'─'*12}  {'─'*5}  {'─'*10}")
for tick, tid, name, work, rem in log[:10]:
    print(f"  {tick:5d}  {tid:5d}  {name:<12}  {work:5d}  {rem:10d}")


## Step 3 — Priority analysis: which tasks got the most CPU?

In [ ]:
from collections import defaultdict

work_by_task  = defaultdict(int)
ticks_by_task = defaultdict(int)
for tick, tid, name, work, rem in log:
    work_by_task[tid]  += work
    ticks_by_task[tid] += 1

total_work = sum(work_by_task.values())

print("CPU allocation by task:")
print(f"  {'ID':>3}  {'Name':<12}  {'Total work':>11}  {'Turns':>6}  {'Share %':>8}")
print(f"  {'-'*3}  {'-'*12}  {'-'*11}  {'-'*6}  {'-'*8}")

for tid, name, _ in completion_order:
    tw  = work_by_task[tid]
    trn = ticks_by_task[tid]
    pct = tw / total_work * 100
    print(f"  {tid:3d}  {name:<12}  {tw:11d}  {trn:6d}  {pct:7.1f}%")

most_cpu_tid  = max(work_by_task, key=work_by_task.get)
most_cpu_name = next(name for _, tid, name, *_ in log if tid == most_cpu_tid)
most_cpu_work = work_by_task[most_cpu_tid]
print()
print(f"  Most CPU: Task {most_cpu_tid} ({most_cpu_name}) — "
      f"{most_cpu_work} units ({most_cpu_work/total_work*100:.1f}%)")


## Step 4 — Find the task that would survive longest (Josephus-style analysis)

In [ ]:
def last_standing(task_ids, step):
    """
    Simulate elimination: every `step`-th task is removed.
    Uses index arithmetic (pure Python, no linked list rebuild needed here)
    but describes the circular list behaviour.
    Returns the ID of the survivor.
    """
    ids    = list(task_ids)
    idx    = 0
    while len(ids) > 1:
        idx = (idx + step - 1) % len(ids)
        ids.pop(idx)
        if idx == len(ids):
            idx = 0
    return ids[0]

original_ids   = [t.task_id for t in circular_to_list(head)] if head else [t.task_id for _, t, *_ in [(None, type('',(),{'task_id':tid})()) for tid,_,_ in completion_order]]
all_task_ids   = [tid for tid, _, _ in completion_order]
step           = random.randint(2, 4)
survivor_id    = last_standing(all_task_ids, step)
survivor_name  = next(n for _, tid, n, *_ in log if tid == survivor_id)

print(f"Josephus analysis (every {step}-th task eliminated):")
print(f"  Tasks         : {all_task_ids}")
print(f"  Step          : {step}")
print(f"  Survivor      : Task {survivor_id} ({survivor_name})")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " ROUND-ROBIN SCHEDULER — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  SCHEDULER PARAMETERS" + " "*(W-22) + "║")
print(f"║  {'Tasks':<28}: {N_TASKS:<26}║")
print(f"║  {'Quantum':<28}: {QUANTUM} work units/turn{'':<18}║")
print(f"║  {'Task names':<28}: {', '.join(selected)[:26]:<26}║")
print("╠" + "═"*W + "╣")
print("║  SIMULATION RESULTS" + " "*(W-20) + "║")
print(f"║  {'Total ticks':<28}: {total_ticks:<26}║")
print(f"║  {'Tasks completed':<28}: {len(completion_order):<26}║")
print(f"║  {'Total work processed':<28}: {total_work:<26}║")
print(f"║  {'Most CPU task':<28}: {most_cpu_name} ({most_cpu_work} units){'':<8}║")
print("╠" + "═"*W + "╣")
print("║  JOSEPHUS ANALYSIS" + " "*(W-19) + "║")
print(f"║  {'Elimination step':<28}: every {step}-th{'':<21}║")
print(f"║  {'Survivor task':<28}: {survivor_name} (ID {survivor_id}){'':<15}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which linked list concept did you find most important, and why?**
Pick one technique from the notebook (fast/slow pointers, dummy head, reversal, cycle detection…) and explain in your own words what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
